In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 10.1 - Overview, paths, and fixed analysis settings
# Purpose:
# Build an unbiased whole-chromosome unitig representation from the
# 176 verified chromosome-only FASTAs produced by Notebook 09.
#
# Workflow:
# 176 chromosome FASTAs
# -> compacted coloured de Bruijn graph
# -> unitigs
# -> remove only invariant unitigs
# -> 176 x M variable-unitig presence/absence matrix
#
# This notebook stops after unitig generation and QC.
# It does NOT test association with ceftazidime MIC.
#
# Storage policy:
# - chromosome FASTAs already in project storage are treated as read-only inputs;
# - temporary copies and unitig-caller working files are kept in /content;
# - only the final compact unitig representation is retained in project storage;
# - no full temporary graph or unfiltered unitig-call file is retained in project storage.
#
# Fixed sequence setting:
# - k = 31, set before any MIC association analysis.

from pathlib import Path
import csv
import gzip
import json
import os
import shutil
import subprocess
import tarfile
import time
from array import array

import numpy as np
import pandas as pd
from IPython.display import display
PROJECT_ROOT = _repo_root()

NOTEBOOK_DIR = (
    PROJECT_ROOT
    / "03_Notebooks"
    / "04_Genome_Comparison"
)

INTERMEDIATE_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "10_Whole_Chromosome_Unitigs"
)

RESULTS_TABLE_DIR = (
    PROJECT_ROOT
    / "05_Results"
    / "Tables"
)

SOURCE_MANIFEST = (
    RESULTS_TABLE_DIR
    / "09_platon_176_chromosome_plasmid_manifest.csv"
)

UNITIG_FASTA = (
    INTERMEDIATE_DIR
    / "10_variable_unitigs.fasta.gz"
)

UNITIG_METADATA = (
    INTERMEDIATE_DIR
    / "10_variable_unitig_metadata.csv.gz"
)

UNITIG_MATRIX = (
    INTERMEDIATE_DIR
    / "10_variable_unitig_matrix_176xM.npz"
)

UNITIG_SAMPLES = (
    INTERMEDIATE_DIR
    / "10_unitig_sample_order.csv"
)

SOFTWARE_VERSIONS = (
    INTERMEDIATE_DIR
    / "10_unitig_software_versions.txt"
)

RUN_LOG = (
    INTERMEDIATE_DIR
    / "10_unitig_caller.log"
)

REPRESENTATION_SUMMARY = (
    RESULTS_TABLE_DIR
    / "10_unitig_representation_summary.csv"
)

FREQUENCY_SUMMARY = (
    RESULTS_TABLE_DIR
    / "10_unitig_presence_frequency_summary.csv"
)

FINAL_QC_FILE = (
    RESULTS_TABLE_DIR
    / "10_unitig_representation_final_QC.csv"
)

COMPLETION_FILE = (
    INTERMEDIATE_DIR
    / "10_UNITIG_REPRESENTATION_COMPLETE.json"
)

KMER_SIZE = 31
N_EXPECTED = 176

for path in [
    PROJECT_ROOT,
    NOTEBOOK_DIR,
    RESULTS_TABLE_DIR,
]:
    assert path.exists(), f"Required path not found: {path}"

INTERMEDIATE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Notebook 10 - Whole-chromosome unitig representation")
print("Notebook folder:", NOTEBOOK_DIR)
print("Output folder:", INTERMEDIATE_DIR)
print("Fixed k-mer size:", KMER_SIZE)
print()
print("No MIC association testing is performed in this notebook.")
print()
print("Transition: Cell 10.2 will verify the 176 chromosome-only FASTAs.")


In [ ]:
#@title Cell 10.2 - Verify the 176 chromosome-only FASTAs
# Purpose:
# Confirm that Notebook 09 produced one verified chromosome FASTA for each
# of the 176 blaTEM-1-only pathogens before unitig construction begins.

assert SOURCE_MANIFEST.exists(), (
    f"Notebook 09 manifest not found: {SOURCE_MANIFEST}"
)

manifest = pd.read_csv(
    SOURCE_MANIFEST
)

required_columns = {
    "biosample",
    "assembly_accession",
    "log2_mic",
    "chromosome_fasta",
    "complete",
}

missing_columns = (
    required_columns
    - set(manifest.columns)
)

assert not missing_columns, (
    "Notebook 09 manifest is missing required columns: "
    + ", ".join(sorted(missing_columns))
)

assert len(manifest) == N_EXPECTED, (
    f"Expected {N_EXPECTED} rows, found {len(manifest)}."
)

assert manifest["biosample"].nunique() == N_EXPECTED, (
    "BioSample identifiers are not unique."
)

assert manifest["assembly_accession"].nunique() == N_EXPECTED, (
    "Assembly accessions are not unique."
)

complete_text = (
    manifest["complete"]
    .astype(str)
    .str.strip()
    .str.lower()
)

assert complete_text.isin(
    {"true", "1"}
).all(), (
    "At least one Notebook 09 row is not marked complete."
)

manifest["chromosome_fasta"] = (
    manifest["chromosome_fasta"]
    .astype(str)
)

manifest["chromosome_fasta_exists"] = (
    manifest["chromosome_fasta"]
    .map(lambda x: Path(x).is_file())
)

manifest["chromosome_fasta_bytes"] = (
    manifest["chromosome_fasta"]
    .map(
        lambda x: Path(x).stat().st_size
        if Path(x).is_file()
        else 0
    )
)

missing_fasta = manifest.loc[
    ~manifest["chromosome_fasta_exists"]
    | (manifest["chromosome_fasta_bytes"] <= 0)
].copy()

if not missing_fasta.empty:
    display(
        missing_fasta[
            [
                "biosample",
                "assembly_accession",
                "chromosome_fasta",
            ]
        ]
    )

assert missing_fasta.empty, (
    "At least one chromosome FASTA is missing or empty."
)

manifest = (
    manifest
    .sort_values("biosample")
    .reset_index(drop=True)
)

print(
    "Verified chromosome FASTAs:",
    len(manifest),
    "/",
    N_EXPECTED,
)

print(
    "Total chromosome FASTA size:",
    f"{manifest['chromosome_fasta_bytes'].sum() / 1024**3:.3f} GiB",
)

display(
    manifest[
        [
            "biosample",
            "assembly_accession",
            "log2_mic",
            "chromosome_fasta",
        ]
    ].head()
)

print("\nCell 10.2 complete.")
print(
    "Transition: Cell 10.3 will install unitig-caller "
    "in temporary Colab storage."
)


In [ ]:
#@title Cell 10.3 - Install unitig-caller in temporary Colab storage
# Purpose:
# Install unitig-caller and its Bifrost dependencies in an isolated
# temporary micromamba environment.
#
# Nothing from the software environment is stored in project storage.

MICROMAMBA_ROOT = Path(
    "/content/micromamba"
)

MICROMAMBA_BIN = (
    MICROMAMBA_ROOT
    / "bin"
    / "micromamba"
)

UNITIG_ENV = Path(
    "/content/nb10_unitig_env"
)

if not MICROMAMBA_BIN.exists():
    print("Installing micromamba in temporary Colab storage...")

    MICROMAMBA_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    archive_path = Path(
        "/content/micromamba.tar.bz2"
    )

    subprocess.run(
        [
            "curl",
            "-L",
            "--fail",
            "--show-error",
            "https://micro.mamba.pm/api/"
            "micromamba/linux-64/latest",
            "-o",
            str(archive_path),
        ],
        check=True,
    )

    with tarfile.open(
        archive_path,
        "r:bz2",
    ) as archive:
        member = archive.getmember(
            "bin/micromamba"
        )
        archive.extract(
            member,
            path=MICROMAMBA_ROOT,
            filter="data",
        )

    MICROMAMBA_BIN.chmod(0o755)

    if archive_path.exists():
        archive_path.unlink()

if not UNITIG_ENV.exists():
    print("Installing unitig-caller and dependencies...")

    subprocess.run(
        [
            str(MICROMAMBA_BIN),
            "create",
            "-y",
            "-p",
            str(UNITIG_ENV),
            "-c",
            "conda-forge",
            "-c",
            "bioconda",
            "unitig-caller",
        ],
        check=True,
    )
else:
    print("Existing temporary unitig-caller environment found.")

def unitig_env_run(
    *args,
    check=True,
    capture_output=False,
):
    return subprocess.run(
        [
            str(MICROMAMBA_BIN),
            "run",
            "-p",
            str(UNITIG_ENV),
            *map(str, args),
        ],
        check=check,
        capture_output=capture_output,
        text=True,
    )

version_result = unitig_env_run(
    "unitig-caller",
    "--version",
    capture_output=True,
)

UNITIG_CALLER_VERSION = (
    version_result.stdout.strip()
    or version_result.stderr.strip()
)

print(
    "unitig-caller version:",
    UNITIG_CALLER_VERSION,
)

help_result = unitig_env_run(
    "unitig-caller",
    "--help",
    capture_output=True,
)

assert "--call" in help_result.stdout
assert "--refs" in help_result.stdout
assert "--pyseer" in help_result.stdout
assert "--kmer" in help_result.stdout

print("\nCell 10.3 complete.")
print(
    "Transition: Cell 10.4 will prepare temporary local "
    "copies of the 176 chromosome FASTAs."
)


In [ ]:
#@title Cell 10.4 - Prepare temporary local chromosome inputs
# Purpose:
# Copy the 176 chromosome FASTAs from Drive to temporary Colab storage.
# This avoids repeated Google Drive I/O during graph construction.
#
# These are temporary copies only and will be deleted after successful
# unitig construction.

LOCAL_INPUT_DIR = Path(
    "/content/nb10_chromosome_inputs"
)

WORK_DIR = Path(
    "/content/nb10_unitig_work"
)

LOCAL_FINAL_DIR = Path(
    "/content/nb10_unitig_final"
)

REFS_FILE = (
    WORK_DIR
    / "10_refs_176.txt"
)

for path in [
    LOCAL_INPUT_DIR,
    WORK_DIR,
    LOCAL_FINAL_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

total, used, free = shutil.disk_usage(
    "/content"
)

print(
    "Free Colab disk before copying inputs:",
    f"{free / 1024**3:.1f} GiB",
)

if free < 10 * 1024**3:
    raise RuntimeError(
        "Less than 10 GiB of free temporary Colab disk remains. "
        "Stop before building the graph."
    )

local_rows = []

for i, row in manifest.iterrows():
    biosample = str(
        row["biosample"]
    )

    source = Path(
        row["chromosome_fasta"]
    )

    destination = (
        LOCAL_INPUT_DIR
        / f"{biosample}.fasta"
    )

    if (
        not destination.exists()
        or destination.stat().st_size
        != source.stat().st_size
    ):
        shutil.copy2(
            source,
            destination,
        )

    local_rows.append(
        {
            "sample_index": i,
            "biosample": biosample,
            "assembly_accession": row[
                "assembly_accession"
            ],
            "log2_mic": row[
                "log2_mic"
            ],
            "chromosome_fasta": str(
                row["chromosome_fasta"]
            ),
            "local_fasta": str(
                destination
            ),
        }
    )

local_manifest = pd.DataFrame(
    local_rows
)

assert len(local_manifest) == N_EXPECTED
assert local_manifest["biosample"].nunique() == N_EXPECTED

for path_text in local_manifest["local_fasta"]:
    path = Path(path_text)
    assert path.is_file() and path.stat().st_size > 0

with open(
    REFS_FILE,
    "w",
    encoding="utf-8",
) as handle:
    for path_text in local_manifest["local_fasta"]:
        handle.write(
            f"{path_text}\n"
        )

local_manifest.to_csv(
    UNITIG_SAMPLES,
    index=False,
)

total, used, free = shutil.disk_usage(
    "/content"
)

print(
    "Temporary chromosome FASTAs prepared:",
    len(local_manifest),
)

print(
    "Reference list:",
    REFS_FILE,
)

print(
    "Free Colab disk after copying inputs:",
    f"{free / 1024**3:.1f} GiB",
)

print(
    "Threads available:",
    os.cpu_count(),
)

print("\nCell 10.4 complete.")
print(
    "Transition: Cell 10.5 will build the unitigs and convert "
    "them directly to the final compact representation."
)


In [ ]:
#@title Cell 10.5 - Build unitigs and create the compact 176 x M representation
# Purpose:
# Run unitig-caller in call mode on the 176 chromosome-only assemblies,
# then immediately convert its temporary output into:
# - variable-unitig FASTA;
# - variable-unitig metadata;
# - 176 x M sparse presence/absence matrix;
# - compact summary tables.
#
# Only unitigs present in all 176 pathogens are removed.
# No MIC-based or biological candidate filtering is performed.
#
# Restart behavior:
# - if a completion marker already exists, this long step is skipped;
# - if interrupted before completion, rerun this cell from the start;
# - partial Drive outputs are never treated as complete.

from scipy import sparse

THREADS = max(
    1,
    min(
        4,
        os.cpu_count() or 2,
    ),
)

OUT_PREFIX = (
    WORK_DIR
    / "unitig_calls"
)

LOCAL_LOG = (
    WORK_DIR
    / "10_unitig_caller.log"
)

LOCAL_FASTA = (
    LOCAL_FINAL_DIR
    / "10_variable_unitigs.fasta.gz"
)

LOCAL_METADATA = (
    LOCAL_FINAL_DIR
    / "10_variable_unitig_metadata.csv.gz"
)

LOCAL_MATRIX = (
    LOCAL_FINAL_DIR
    / "10_variable_unitig_matrix_176xM.npz"
)

LOCAL_REPRESENTATION_SUMMARY = (
    LOCAL_FINAL_DIR
    / "10_unitig_representation_summary.csv"
)

LOCAL_FREQUENCY_SUMMARY = (
    LOCAL_FINAL_DIR
    / "10_unitig_presence_frequency_summary.csv"
)

LOCAL_VERSIONS = (
    LOCAL_FINAL_DIR
    / "10_unitig_software_versions.txt"
)

required_final_outputs = [
    UNITIG_FASTA,
    UNITIG_METADATA,
    UNITIG_MATRIX,
    UNITIG_SAMPLES,
    REPRESENTATION_SUMMARY,
    FREQUENCY_SUMMARY,
    SOFTWARE_VERSIONS,
    RUN_LOG,
]

if COMPLETION_FILE.exists():
    print(
        "Completion marker already exists."
    )

    missing_final = [
        str(path)
        for path in required_final_outputs
        if not path.exists()
    ]

    assert not missing_final, (
        "Completion marker exists but final output(s) are missing:\n"
        + "\n".join(missing_final)
    )

    print(
        "All final unitig outputs are present. "
        "Long construction step skipped."
    )

else:
    if LOCAL_FINAL_DIR.exists():
        shutil.rmtree(
            LOCAL_FINAL_DIR
        )

    LOCAL_FINAL_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    for path in WORK_DIR.glob(
        "unitig_calls*"
    ):
        if path.is_file():
            path.unlink()
        elif path.is_dir():
            shutil.rmtree(
                path,
                ignore_errors=True,
            )

    print(
        "Running unitig-caller on",
        N_EXPECTED,
        "chromosome FASTAs",
    )

    print(
        "k-mer size:",
        KMER_SIZE,
    )

    print(
        "threads:",
        THREADS,
    )

    start_time = time.time()

    command = [
        str(MICROMAMBA_BIN),
        "run",
        "-p",
        str(UNITIG_ENV),
        "unitig-caller",
        "--call",
        "--refs",
        str(REFS_FILE),
        "--out",
        str(OUT_PREFIX),
        "--pyseer",
        "--kmer",
        str(KMER_SIZE),
        "--threads",
        str(THREADS),
    ]

    with open(
        LOCAL_LOG,
        "w",
        encoding="utf-8",
    ) as log_handle:
        result = subprocess.run(
            command,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            text=True,
        )

    elapsed_build_minutes = (
        time.time()
        - start_time
    ) / 60

    print(
        "unitig-caller elapsed:",
        f"{elapsed_build_minutes:.1f} minutes",
    )

    if result.returncode != 0:
        tail = (
            LOCAL_LOG.read_text(
                errors="replace"
            )
            .splitlines()[-40:]
        )

        shutil.copy2(
            LOCAL_LOG,
            RUN_LOG,
        )

        raise RuntimeError(
            "unitig-caller failed.\nLast log lines:\n"
            + "\n".join(tail)
        )

    expected_pyseer = Path(
        str(OUT_PREFIX)
        + ".pyseer"
    )

    if expected_pyseer.exists():
        PYSEER_FILE = expected_pyseer
    else:
        candidates = sorted(
            WORK_DIR.glob(
                "unitig_calls*pyseer*"
            )
        )

        assert len(candidates) == 1, (
            "Could not identify exactly one unitig-caller "
            "pyseer output. Files found:\n"
            + "\n".join(
                str(x)
                for x in sorted(
                    WORK_DIR.glob(
                        "unitig_calls*"
                    )
                )
            )
        )

        PYSEER_FILE = candidates[0]

    assert PYSEER_FILE.is_file()
    assert PYSEER_FILE.stat().st_size > 0

    print(
        "Temporary unitig-call file:",
        PYSEER_FILE,
    )

    print(
        "Temporary unitig-call size:",
        f"{PYSEER_FILE.stat().st_size / 1024**3:.3f} GiB",
    )

    sample_names = (
        local_manifest[
            "biosample"
        ]
        .astype(str)
        .tolist()
    )

    sample_to_index = {
        sample: i
        for i, sample in enumerate(
            sample_names
        )
    }

    def token_to_biosample(token):
        value = token.strip()

        if value.endswith(":1"):
            value = value[:-2]

        base = Path(
            value
        ).name

        for suffix in [
            ".chromosome.fasta",
            ".chromosome.fa",
            ".chromosome.fna",
            ".fasta",
            ".fa",
            ".fna",
        ]:
            if base.endswith(
                suffix
            ):
                base = base[
                    :-len(suffix)
                ]
                break

        return base

    row_indices = array(
        "B"
    )

    col_indices = array(
        "I"
    )

    variable_lengths = array(
        "I"
    )

    frequency_counts = np.zeros(
        N_EXPECTED + 1,
        dtype=np.int64,
    )

    total_unitigs = 0
    invariant_present = 0
    variable_unitigs = 0
    nnz = 0

    start_parse = time.time()

    with gzip.open(
        LOCAL_FASTA,
        "wt",
        encoding="utf-8",
    ) as fasta_handle, gzip.open(
        LOCAL_METADATA,
        "wt",
        encoding="utf-8",
        newline="",
    ) as metadata_handle:

        metadata_writer = csv.writer(
            metadata_handle
        )

        metadata_writer.writerow(
            [
                "unitig_index",
                "unitig_id",
                "length_bp",
                "present_count",
                "present_fraction",
            ]
        )

        with open(
            PYSEER_FILE,
            "r",
            encoding="utf-8",
            errors="replace",
        ) as input_handle:

            for line_number, raw_line in enumerate(
                input_handle,
                start=1,
            ):
                line = raw_line.strip()

                if not line:
                    continue

                total_unitigs += 1

                if "|" not in line:
                    raise RuntimeError(
                        f"Unexpected pyseer format at line {line_number}."
                    )

                sequence_text, presence_text = (
                    line.split(
                        "|",
                        1,
                    )
                )

                sequence = (
                    sequence_text
                    .strip()
                    .upper()
                )

                if not sequence:
                    raise RuntimeError(
                        f"Empty unitig sequence at line {line_number}."
                    )

                present_samples = set()

                for token in presence_text.strip().split():
                    biosample = token_to_biosample(
                        token
                    )

                    if biosample not in sample_to_index:
                        raise RuntimeError(
                            "Unknown sample name in unitig-caller output "
                            f"at line {line_number}: {biosample}"
                        )

                    present_samples.add(
                        biosample
                    )

                present_count = len(
                    present_samples
                )

                if (
                    present_count < 1
                    or present_count > N_EXPECTED
                ):
                    raise RuntimeError(
                        "Invalid unitig presence count "
                        f"{present_count} at line {line_number}."
                    )

                frequency_counts[
                    present_count
                ] += 1

                if present_count == N_EXPECTED:
                    invariant_present += 1
                    continue

                unitig_index = (
                    variable_unitigs
                )

                unitig_id = (
                    f"U{unitig_index + 1:09d}"
                )

                fasta_handle.write(
                    f">{unitig_id}\n"
                )

                for start in range(
                    0,
                    len(sequence),
                    80,
                ):
                    fasta_handle.write(
                        sequence[
                            start:start + 80
                        ]
                        + "\n"
                    )

                metadata_writer.writerow(
                    [
                        unitig_index,
                        unitig_id,
                        len(sequence),
                        present_count,
                        present_count
                        / N_EXPECTED,
                    ]
                )

                variable_lengths.append(
                    len(sequence)
                )

                sample_indices = sorted(
                    sample_to_index[
                        sample
                    ]
                    for sample in present_samples
                )

                row_indices.extend(
                    sample_indices
                )

                col_indices.extend(
                    array(
                        "I",
                        [unitig_index]
                    )
                    * present_count
                )

                nnz += present_count
                variable_unitigs += 1

    assert (
        total_unitigs
        == invariant_present
        + variable_unitigs
    )

    assert variable_unitigs > 0

    assert variable_unitigs < 2_147_483_647, (
        "Variable-unitig count exceeds 32-bit sparse-matrix indexing."
    )

    print(
        "Total unitigs:",
        f"{total_unitigs:,}",
    )

    print(
        "Invariant unitigs:",
        f"{invariant_present:,}",
    )

    print(
        "Variable unitigs M:",
        f"{variable_unitigs:,}",
    )

    print(
        "Presence entries:",
        f"{nnz:,}",
    )

    rows_np = np.frombuffer(
        row_indices,
        dtype=np.uint8,
    ).astype(
        np.int32
    )

    cols_np = np.frombuffer(
        col_indices,
        dtype=np.uint32,
    ).astype(
        np.int32
    )

    data_np = np.ones(
        len(rows_np),
        dtype=np.uint8,
    )

    matrix = sparse.csc_matrix(
        (
            data_np,
            (
                rows_np,
                cols_np,
            ),
        ),
        shape=(
            N_EXPECTED,
            variable_unitigs,
        ),
        dtype=np.uint8,
    )

    matrix.sum_duplicates()

    if matrix.nnz:
        matrix.data[:] = 1

    assert matrix.shape == (
        N_EXPECTED,
        variable_unitigs,
    )

    assert matrix.nnz == nnz, (
        "Sparse-matrix nonzero count differs from parsed unitig calls."
    )

    sparse.save_npz(
        LOCAL_MATRIX,
        matrix,
        compressed=True,
    )

    variable_lengths_np = np.frombuffer(
        variable_lengths,
        dtype=np.uint32,
    )

    density = (
        matrix.nnz
        / (
            N_EXPECTED
            * variable_unitigs
        )
    )

    sparse_raw_estimate = (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    )

    summary_row = {
        "n_pathogens": N_EXPECTED,
        "kmer_size": KMER_SIZE,
        "total_unitigs": total_unitigs,
        "invariant_present_in_176": invariant_present,
        "variable_unitigs_M": variable_unitigs,
        "matrix_rows": N_EXPECTED,
        "matrix_columns": variable_unitigs,
        "matrix_nonzero_entries": matrix.nnz,
        "matrix_density": density,
        "unitig_length_min_bp": int(
            variable_lengths_np.min()
        ),
        "unitig_length_median_bp": float(
            np.median(
                variable_lengths_np
            )
        ),
        "unitig_length_mean_bp": float(
            variable_lengths_np.mean()
        ),
        "unitig_length_max_bp": int(
            variable_lengths_np.max()
        ),
        "dense_uint8_matrix_estimate_GiB": (
            N_EXPECTED
            * variable_unitigs
            / 1024**3
        ),
        "sparse_raw_estimate_GiB": (
            sparse_raw_estimate
            / 1024**3
        ),
        "unitig_caller_elapsed_minutes": (
            elapsed_build_minutes
        ),
        "unitig_parse_elapsed_minutes": (
            (time.time() - start_parse)
            / 60
        ),
    }

    pd.DataFrame(
        [summary_row]
    ).to_csv(
        LOCAL_REPRESENTATION_SUMMARY,
        index=False,
    )

    frequency_table = pd.DataFrame(
        {
            "present_count": np.arange(
                1,
                N_EXPECTED + 1,
            ),
            "n_unitigs": frequency_counts[
                1:
            ],
        }
    )

    frequency_table[
        "present_fraction"
    ] = (
        frequency_table[
            "present_count"
        ]
        / N_EXPECTED
    )

    frequency_table.to_csv(
        LOCAL_FREQUENCY_SUMMARY,
        index=False,
    )

    version_text = (
        f"unitig-caller: {UNITIG_CALLER_VERSION}\n"
        f"k-mer size: {KMER_SIZE}\n"
        f"threads: {THREADS}\n"
    )

    LOCAL_VERSIONS.write_text(
        version_text,
        encoding="utf-8",
    )

    output_pairs = [
        (
            LOCAL_FASTA,
            UNITIG_FASTA,
        ),
        (
            LOCAL_METADATA,
            UNITIG_METADATA,
        ),
        (
            LOCAL_MATRIX,
            UNITIG_MATRIX,
        ),
        (
            LOCAL_REPRESENTATION_SUMMARY,
            REPRESENTATION_SUMMARY,
        ),
        (
            LOCAL_FREQUENCY_SUMMARY,
            FREQUENCY_SUMMARY,
        ),
        (
            LOCAL_VERSIONS,
            SOFTWARE_VERSIONS,
        ),
        (
            LOCAL_LOG,
            RUN_LOG,
        ),
    ]

    for source, destination in output_pairs:
        partial = Path(
            str(destination)
            + ".partial"
        )

        if partial.exists():
            partial.unlink()

        shutil.copy2(
            source,
            partial,
        )

        os.replace(
            partial,
            destination,
        )

    completion_payload = {
        "status": "complete",
        "n_pathogens": N_EXPECTED,
        "kmer_size": KMER_SIZE,
        "total_unitigs": int(
            total_unitigs
        ),
        "variable_unitigs_M": int(
            variable_unitigs
        ),
        "matrix_nonzero_entries": int(
            matrix.nnz
        ),
    }

    partial_completion = Path(
        str(COMPLETION_FILE)
        + ".partial"
    )

    partial_completion.write_text(
        json.dumps(
            completion_payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    os.replace(
        partial_completion,
        COMPLETION_FILE,
    )

    print(
        "\nFinal compact outputs saved to project storage."
    )

    for path in [
        LOCAL_INPUT_DIR,
        WORK_DIR,
        LOCAL_FINAL_DIR,
    ]:
        if path.exists():
            shutil.rmtree(
                path,
                ignore_errors=True,
            )

    print(
        "Temporary chromosome copies and unitig-caller "
        "working files deleted."
    )

print("\nCell 10.5 complete.")
print(
    "Transition: Cell 10.6 will independently verify the "
    "saved 176 x M representation."
)


In [ ]:
#@title Cell 10.6 - Independent QC of the saved unitig representation
# Purpose:
# Independently verify the final unitig FASTA, metadata, sample order,
# sparse matrix, and presence-frequency counts.
#
# This cell can be run after a Colab restart using only Cells 10.1-10.2.

from scipy import sparse

required_outputs = [
    UNITIG_FASTA,
    UNITIG_METADATA,
    UNITIG_MATRIX,
    UNITIG_SAMPLES,
    REPRESENTATION_SUMMARY,
    FREQUENCY_SUMMARY,
    COMPLETION_FILE,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not path.exists()
]

assert not missing_outputs, (
    "Required final output(s) missing:\n"
    + "\n".join(
        missing_outputs
    )
)

samples = pd.read_csv(
    UNITIG_SAMPLES
)

assert len(samples) == N_EXPECTED
assert samples["biosample"].nunique() == N_EXPECTED

matrix = sparse.load_npz(
    UNITIG_MATRIX
).tocsc()

assert matrix.shape[0] == N_EXPECTED

M = matrix.shape[1]

assert M > 0

column_sums = np.asarray(
    matrix.sum(
        axis=0
    )
).ravel()

assert len(
    column_sums
) == M

assert column_sums.min() >= 1
assert column_sums.max() <= (
    N_EXPECTED - 1
)

metadata_rows = 0
metadata_mismatch = 0
metadata_ids = set()

with gzip.open(
    UNITIG_METADATA,
    "rt",
    encoding="utf-8",
    newline="",
) as handle:

    reader = csv.DictReader(
        handle
    )

    expected_fields = {
        "unitig_index",
        "unitig_id",
        "length_bp",
        "present_count",
        "present_fraction",
    }

    assert expected_fields.issubset(
        set(
            reader.fieldnames
            or []
        )
    )

    for row in reader:
        idx = int(
            row["unitig_index"]
        )

        unitig_id = row[
            "unitig_id"
        ]

        present_count = int(
            row["present_count"]
        )

        assert idx == metadata_rows

        if present_count != int(
            column_sums[idx]
        ):
            metadata_mismatch += 1

        metadata_ids.add(
            unitig_id
        )

        metadata_rows += 1

assert metadata_rows == M
assert len(metadata_ids) == M
assert metadata_mismatch == 0

fasta_records = 0
fasta_ids = set()

with gzip.open(
    UNITIG_FASTA,
    "rt",
    encoding="utf-8",
) as handle:
    for line in handle:
        if line.startswith(">"):
            fasta_records += 1
            fasta_ids.add(
                line[1:]
                .strip()
                .split()[0]
            )

assert fasta_records == M
assert len(fasta_ids) == M
assert fasta_ids == metadata_ids

frequency_from_matrix = np.bincount(
    column_sums.astype(
        np.int64
    ),
    minlength=N_EXPECTED + 1,
)

frequency_saved = pd.read_csv(
    FREQUENCY_SUMMARY
)

saved_counts = np.zeros(
    N_EXPECTED + 1,
    dtype=np.int64,
)

for row in frequency_saved.itertuples(
    index=False
):
    saved_counts[
        int(row.present_count)
    ] = int(
        row.n_unitigs
    )

assert np.array_equal(
    frequency_from_matrix[
        1:N_EXPECTED
    ],
    saved_counts[
        1:N_EXPECTED
    ],
)

summary = pd.read_csv(
    REPRESENTATION_SUMMARY
)

assert len(summary) == 1
assert int(
    summary.loc[
        0,
        "variable_unitigs_M",
    ]
) == M

assert int(
    summary.loc[
        0,
        "matrix_nonzero_entries",
    ]
) == matrix.nnz

qc_row = {
    "n_pathogens": N_EXPECTED,
    "variable_unitigs_M": M,
    "matrix_shape": (
        f"{matrix.shape[0]} x {matrix.shape[1]}"
    ),
    "matrix_nonzero_entries": int(
        matrix.nnz
    ),
    "matrix_density": float(
        matrix.nnz
        / (
            matrix.shape[0]
            * matrix.shape[1]
        )
    ),
    "minimum_unitig_presence": int(
        column_sums.min()
    ),
    "maximum_unitig_presence": int(
        column_sums.max()
    ),
    "metadata_rows": metadata_rows,
    "fasta_records": fasta_records,
    "metadata_matrix_agreement": (
        metadata_mismatch == 0
    ),
    "all_variable": bool(
        column_sums.min() >= 1
        and column_sums.max()
        <= N_EXPECTED - 1
    ),
    "final_QC_pass": True,
}

pd.DataFrame(
    [qc_row]
).to_csv(
    FINAL_QC_FILE,
    index=False,
)

print(
    "Verified pathogens:",
    N_EXPECTED,
)

print(
    "Variable unitigs M:",
    f"{M:,}",
)

print(
    "Matrix shape:",
    f"{matrix.shape[0]} x {matrix.shape[1]:,}",
)

print(
    "Nonzero entries:",
    f"{matrix.nnz:,}",
)

print(
    "Matrix density:",
    f"{qc_row['matrix_density']:.4f}",
)

print(
    "Unitig presence range:",
    f"{int(column_sums.min())} to {int(column_sums.max())} pathogens",
)

print(
    "FASTA / metadata / matrix agreement:",
    "PASS",
)

print(
    "\nSaved final QC:",
    FINAL_QC_FILE,
)

print("\nCell 10.6 complete.")
print(
    "Transition: Cell 10.7 will summarize the representation "
    "and close Notebook 10."
)


In [ ]:
#@title Cell 10.7 - Final summary and stopping point
# Purpose:
# Report the final whole-chromosome unitig representation.
# No MIC association analysis is performed here.

summary = pd.read_csv(
    REPRESENTATION_SUMMARY
)

qc = pd.read_csv(
    FINAL_QC_FILE
)

frequency = pd.read_csv(
    FREQUENCY_SUMMARY
)

display(
    summary.T.rename(
        columns={
            0: "value"
        }
    )
)

print(
    "\nVariable-unitig presence-frequency distribution:"
)

display(
    frequency.loc[
        frequency["n_unitigs"] > 0,
        [
            "present_count",
            "present_fraction",
            "n_unitigs",
        ],
    ]
)

print(
    "\nFinal QC:",
    qc.loc[
        0,
        "final_QC_pass",
    ],
)

print(
    "\nRetained unitig files:"
)

for path in [
    UNITIG_FASTA,
    UNITIG_METADATA,
    UNITIG_MATRIX,
    UNITIG_SAMPLES,
]:
    print(
        "-",
        path.name,
        f"({path.stat().st_size / 1024**2:.1f} MiB)",
    )

assert bool(
    qc.loc[
        0,
        "final_QC_pass",
    ]
)

print(
    "\nFINAL STATUS: whole-chromosome variable-unitig "
    "representation accepted."
)

print(
    "\nNotebook 10 ends here."
)

print(
    "Do not begin unitig-MIC association testing until "
    "this representation and its M value have been reviewed."
)
